Angle encoding for quantum machine learning

In [1]:
pip install numpy pandas qiskit

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit import QuantumCircuit

In [5]:
file = "../Dataset/final_sdb_dataset_clean.csv"
df = pd.read_csv(file)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (11676, 9)
    latitude  longitude  blue_band  green_band  depth  stumpf_ratio  log_blue  \
0  15.043750  80.877083     0.0462      0.0360   2321      1.069613 -3.074775   
1   9.627083  82.210417     0.0469      0.0344   3709      1.087608 -3.059738   
2   8.377083  76.043750     0.0501      0.0337   1094      1.112729 -2.993734   
3   8.085417  72.127083     0.0489      0.0336   2566      1.106772 -3.017978   
4   9.085417  74.293750     0.0339      0.0244   2732      1.102934 -3.384340   

   log_green  bg_ratio  
0  -3.324236  1.283333  
1  -3.369699  1.363372  
2  -3.390257  1.486647  
3  -3.393229  1.455357  
4  -3.713172  1.389344  


In [6]:
features = [
    "blue_band",
    "green_band",
    "log_blue",
    "log_green",
    "bg_ratio",
    "stumpf_ratio"
]

target = "depth"

X = df[features].to_numpy(dtype=np.float64)
y = df[target].to_numpy(dtype=np.float64)

In [7]:
df = df.replace([np.inf, -np.inf], np.nan)

df = df.dropna(
    subset=features + [target]
).copy()

X = df[features].to_numpy(dtype=np.float64)
y = df[target].to_numpy(dtype=np.float64)

In [8]:
mask = (y >= 0) & (y <= 30)

X = X[mask]
y = y[mask]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [10]:
scaler = MinMaxScaler(feature_range=(0, np.pi))

X_train_angle = scaler.fit_transform(X_train)

X_test_angle = scaler.transform(X_test)

In [11]:
N_QUBITS = len(features)

def angle_encode(x):

    qc = QuantumCircuit(N_QUBITS)

    for i in range(N_QUBITS):
        qc.ry(x[i], i)

    return qc

In [12]:
sample = X_train_angle[0]

qc = angle_encode(sample)

print(qc.draw())

     ┌─────────────┐
q_0: ┤ Ry(0.39155) ├
     ├─────────────┤
q_1: ┤ Ry(0.31608) ├
     └┬────────────┤
q_2: ─┤ Ry(1.5673) ├
      ├────────────┤
q_3: ─┤ Ry(1.4278) ├
      ├────────────┤
q_4: ─┤ Ry(1.4158) ├
      ├───────────┬┘
q_5: ─┤ Ry(1.712) ├─
      └───────────┘ 


In [14]:
print("Original sample:")
print(X_train[0])

print("\nNormalized angles:")
print(X_train_angle[0])

Original sample:
[ 0.0598      0.0555     -2.81674962 -2.89137226  1.07747748  1.01857956]

Normalized angles:
[0.39155196 0.31607905 1.56733069 1.42784234 1.41584492 1.71196003]


In [16]:
angle_features = [
    "blue_angle",
    "green_angle",
    "log_blue_angle",
    "log_green_angle",
    "bg_ratio_angle",
    "stumpf_angle"
]

train_encoded_df = pd.DataFrame(
    X_train_angle,
    columns=angle_features
)

test_encoded_df = pd.DataFrame(
    X_test_angle,
    columns=angle_features
)

# Add target depth
train_encoded_df["depth"] = y_train
test_encoded_df["depth"] = y_test

In [17]:
train_encoded_df.to_csv(
    "train_angle_encoded.csv",
    index=False
)

test_encoded_df.to_csv(
    "test_angle_encoded.csv",
    index=False
)

In [18]:
train_encoded_df.to_csv(
    "train_angle_encoded.csv",
    index=False
)

test_encoded_df.to_csv(
    "test_angle_encoded.csv",
    index=False
)